# Support Vector Machines

**Pick the sepreate line that maximize the distance between the decision boundary adn the nearest data points on the each side**

## Problem definition

Find the bigest margin from the hyperplane to the nearest point on either side

```
minimize    0.5 * ||w||^2

subject to y_i * (w^T * x_i + b) >= 1 for all i
```

## Lose The Margin

Some points may be on the wrong side of the boundary, or inside the margin.
So we allows violiations by introducing slack variabls.

```
minimize  0.5 * ||w||^2 + C * sum(xi_i)
subject to   y_i * (w^T * x_i + b) >= 1 - xi_i, xi_i >= 0 for all i
```

The slack variable xi_i measures how much point i violates the margin. C controls the trade-off

||C value|Behavior|
|---|---|---|
||Large C|Narrow margin, fewer misclassifications, Overfits|
||Small C|Wide margin, more misclassifications, Underfits|

## Rewrite Soft Margin SVM

```
minimize  0.5 * ||w||^2 + C * sum(max(0, 1 - y_i * (w^T * x_i + b)))
```

### Hinge loss

max(0, 1 - ..) means when the point is correctly classified and beyond the margin, penality is zero. else, it's linear when the point is inside the margin or misclassfied.

## Training

hinge loss plus L2 regularization

it's just a matter of how C is defined

```
L(w, b) = (lambda / 2) * ||w||^2 + (1 / n) * sum(max(0, 1 - y_i * (w^T * x_i + b)))

dL/dw = lambda * w                 if score(eg. w^T * x_i + b) >= 1
dL/dw = lambda * w - y_i * x_i     if score < 1

dL/db = 0                          if score >= 1
dL/db = -y_i                       if score < 1

```



## Dual formulation

```
maximize     sum(alpha_i) - 0.5 * sum_ij(alpha_i * alpha_j * y_i * y_j * dot(x_i, y_j))
subject to     0 <= alpha_i <= C   sum(alpha_i * y_i) = 0
```

dot(x_i, x_j) can be replaced with kernel function K(x_i, x_j)

```
Linear kernel:      K(x, z) = dot(x, z)
Polynomial kernel:  K(x, z) = (dot(x, z) + c)^d
RBF(Guassian):      K(x, z) = exp(-gamma * ||x - z||^ 2)
```

## Build your own

In [7]:
import math

def dot(x, y):
    return sum(x_i * y_i for x_i, y_i in zip(x, y))

def hinge_loss(X, y, w, b):
    n = len(X)
    total_loss = 0.0
    for i in range(n):
        margin = y[i] * (dot(w, X[i]) + b)
        total_loss += max(0.0, 1.0 - margin)
    return total_loss / n

In [8]:
class LinearSVM:
    def __init__(self, lr=0.001, lambda_param=0.01, n_epochs=1000):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_epochs = n_epochs
        self.w = None
        self.b = 0.0
        self.loss_history = []

    def fit(self, X, y):
        n_features = len(X[0])
        self.w = [0.0] * n_features
        self.b = 0.0
        self.loss_history = []

        for epoch in range(self.n_epochs):
            for i in range(len(X)):
                margin = y[i] * (dot(self.w, X[i]) + self.b)
                if margin >= 1:
                    self.w = [wj - self.lr * self.lambda_param * wj
                              for wj in self.w]
                else:
                    self.w = [wj - self.lr * (self.lambda_param * wj - y[i] * X[i][j])
                              for j, wj in enumerate(self.w)]
                    self.b -= self.lr * (-y[i])
            if epoch % 100 == 0:
                self.loss_history.append((epoch, hinge_loss(X, y, self.w, self.b)))

    def predict(self, X):
        return [1 if dot(self.w, x) + self.b >= 0 else -1 for x in X]

    def margin_width(self):
        # full margin = 2 / ||w||
        norm = math.sqrt(sum(wj * wj for wj in self.w))
        return 2.0 / norm if norm > 0 else float("inf")


In [9]:
def linear_kernel(x, z):
    return dot(x, z)

def polynomial_kernel(x, z, degree=3, c=1.0):
    return (dot(x, z) + c) ** degree

def rbf_kernel(x, z, gamma=0.5):
    diff = [xi - zi for xi, zi in zip(x, z)]
    return math.exp(-gamma * dot(diff, diff))

In [ ]:
import random
%matplotlib inline
import matplotlib.pyplot as plt

def generate_linear_data(n_samples=100, margin=1.0, seed=42):
    random.seed(seed)
    X = []
    y = []
    for _ in range(n_samples):
        x1 = random.uniform(-3, 3)
        x2 = random.uniform(-3, 3)
        val = x1 + x2
        if val > margin / 2:
            X.append([x1, x2])
            y.append(1)
        elif val < -margin / 2:
            X.append([x1, x2])
            y.append(-1)
    return X, y

def train_test_split(X, y, test_ratio=0.2, seed=42):
    random.seed(seed)
    n = len(X)
    indices = list(range(n))
    random.shuffle(indices)
    split = int(n * (1 - test_ratio))
    train_idx = indices[:split]
    test_idx = indices[split:]
    return (
        [X[i] for i in train_idx],
        [y[i] for i in train_idx],
        [X[i] for i in test_idx],
        [y[i] for i in test_idx],
    )

def demo_linear_svm():
    print("=" * 65)
    print("LINEAR SVM: MAXIMUM MARGIN CLASSIFIER")
    print("=" * 65)
    print()

    X, y = generate_linear_data(200, margin=1.0, seed=42)
    X_train, y_train, X_test, y_test = train_test_split(X, y)

    print(f"  Dataset: {len(X)} samples, linearly separable")
    print(f"  Train: {len(X_train)}  Test: {len(X_test)}")
    print()

    svm = LinearSVM(lr=0.001, lambda_param=0.01, n_epochs=500)
    svm.fit(X_train, y_train)

    train_pred = svm.predict(X_train)
    test_pred = svm.predict(X_test)
    train_acc = sum(p == t for p, t in zip(train_pred, y_train)) / len(y_train)
    test_acc = sum(p == t for p, t in zip(test_pred, y_test)) / len(y_test)

    print(f"  Train accuracy: {train_acc:.4f}")
    print(f"  Test accuracy:  {test_acc:.4f}")
    print(f"  Weights: [{svm.w[0]:.4f}, {svm.w[1]:.4f}]")
    print(f"  Bias: {svm.b:.4f}")
    print(f"  Margin width: {svm.margin_width():.4f}")
    print()

    print("  Training loss progression:")
    print(f"  {'Epoch':>8s}  {'Loss':>10s}")
    print(f"  {'-' * 8}  {'-' * 10}")
    for epoch, loss in svm.loss_history:
        print(f"  {epoch:>8d}  {loss:>10.4f}")
    print()

    ## Draw scatter and margin
    # Decision: w·x + b = 0
    # Margins:  w·x + b = ±1
    x0_vals = [x[0] for x in X]
    x1_vals = [x[1] for x in X]
    x0_min, x0_max = min(x0_vals) - 0.5, max(x0_vals) + 0.5
    x0_line = [x0_min + i * (x0_max - x0_min) / 99 for i in range(100)]
    w0, w1 = svm.w
    decision = [-(w0 * x0 + svm.b) / w1 for x0 in x0_line]
    margin_pos = [-(w0 * x0 + svm.b - 1) / w1 for x0 in x0_line]
    margin_neg = [-(w0 * x0 + svm.b + 1) / w1 for x0 in x0_line]

    plt.figure(figsize=(7, 6))
    plt.scatter(
        x0_vals, x1_vals,
        c=["red" if yi == 1 else "blue" for yi in y],
        edgecolors="k", linewidths=0.3,
    )
    plt.plot(x0_line, decision, color="green", linewidth=2, label="decision (w·x+b=0)")
    plt.plot(x0_line, margin_pos, color="green", linestyle="--", linewidth=1.5, label="margin (w·x+b=±1)")
    plt.plot(x0_line, margin_neg, color="green", linestyle="--", linewidth=1.5)
    plt.fill_between(x0_line, margin_neg, margin_pos, color="green", alpha=0.1)
    plt.xlabel("x0")
    plt.ylabel("x1")
    plt.title("Linear SVM: Scatter + Maximum Margin")
    plt.legend()
    plt.tight_layout()
    plt.show()

demo_linear_svm()
